# 02 Aplicación a los modelos sin métricas

Aplica la configuración optimizada (GRASP) y la de todas las reglas a los modelos cuyo `performance_metrics` no tiene valores numéricos utilizables.

**Procedencia.** Los valores salen del *cuerpo* del README, no del `model-index`. En fichas AutoTrain ambos reportan mediciones distintas (p. ej. ROUGE-2: 12,47 frente a 10,90), así que el resultado se guarda en una columna **aparte** (`metrics_from_body`) y no se mezcla con `performance_metrics`.

**Validez.** Precisión medida: 0,99 en las 390 fichas con referencia; 36 de 36 fichas correctas en la muestra manual sin referencia (IC 95 %: 90,4–100 %), casi toda en formato AutoTrain. En formatos libres la precisión no está medida.

In [1]:
import importlib, time
from pathlib import Path
import pandas as pd

import metric_extraction.gold as gd
import metric_extraction.rules as rl
import metric_extraction.evaluator as ev
import metric_extraction.baselines as bl
import metric_extraction.grasp as gr
import metric_extraction.apply as ap
from metric_extraction.download import safe_name
from metric_extraction.evidence import read_body
from metric_extraction.evaluator import Instance, Configuration
from metric_extraction.gold import gold_tuples

df = pd.read_csv("../data/HFCO2_2026.csv", sep=",")
ref = df[df["performance_metrics"].notna()]

cards = []
for _, r in ref.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    g = gold_tuples(r["performance_metrics"])
    if p.exists() and g:
        cards.append({"id": r["modelId"], "tag": str(r["pipeline_tag"]), "gold": g, "text": read_body(p)})

cat = rl.build_catalog(); n = len(cat)
t0 = time.time(); inst = Instance(cards, cat)
print(f"{len(cards)} fichas de referencia, {n} reglas, precálculo {time.time()-t0:.0f}s")

390 fichas de referencia, 27 reglas, precálculo 66s


## 1. Configuraciones

GRASP (K=6, 80 ms por ficha, mejor de 5 semillas) y, como comparación, todas las reglas en el orden fila, columna, línea (sin límite de K ni de tiempo).

In [2]:
mejor = None
for s in range(5):
    o = bl.Objective(inst, pmin=0.90, tmax=0.080)
    cfg, _ = gr.grasp(o, n, 6, 3000, alpha=0.3, seed=s)
    if mejor is None or o(cfg) > mejor[0]:
        mejor = (o(cfg), cfg, o.metrics(cfg))
f, cfg, m = mejor
print("GRASP:", [cat[i].name for i in cfg.rules], cfg.theta,
      f"| F1 macro={m['macro_F1']:.3f} P={m['P']:.3f} R={m['R']:.3f} t={m['time']*1000:.0f}ms")
ap.save_config(cfg, cat, "../results/config_final.json",
               {"K": 6, "tmax_s": 0.080, "alpha": 0.3, "presupuesto": 3000, "fichas_ref": len(cards)})

todas = Configuration(tuple(range(9, 27)) + tuple(range(0, 9)), 100.0)
mt = bl.Objective(inst, pmin=0.90).metrics(todas)
print("Todas:", f"F1 macro={mt['macro_F1']:.3f} P={mt['P']:.3f} R={mt['R']:.3f} t={mt['time']*1000:.0f}ms")
ap.save_config(todas, cat, "../results/config_todas.json", {"K": 27, "nota": "sin límite de K ni de tiempo"})

GRASP: ['column:clf_basic', 'column:rouge', 'row:sim', 'row:clf_basic', 'inline:rank', 'row:at_k'] 100.0 | F1 macro=0.340 P=0.995 R=0.707 t=79ms
Todas: F1 macro=0.384 P=0.993 R=0.801 t=170ms


## 2. Aplicación a los modelos sin métricas utilizables

In [3]:
rules_g, th_g, _ = ap.load_config("../results/config_final.json")
rules_t, th_t, _ = ap.load_config("../results/config_todas.json")

t0 = time.time()
res_g = ap.apply_to_models(df, "../data/raw/readmes", rules_g, th_g)
res_t = ap.apply_to_models(df, "../data/raw/readmes", rules_t, th_t)
print(f"{len(res_g)} modelos procesados en {time.time()-t0:.0f}s")

Path("../data/processed").mkdir(parents=True, exist_ok=True)
res_g.to_csv("../data/processed/metrics_from_body_grasp.csv", index=False)
res_t.to_csv("../data/processed/metrics_from_body_todas.csv", index=False)

comp = pd.DataFrame({"GRASP (K=6)": res_g["estado"].value_counts(),
                     "Todas las reglas": res_t["estado"].value_counts()}).fillna(0).astype(int)
display(comp)

1664 modelos procesados en 6s


,GRASP (K=6),Todas las reglas
estado,,
con_metricas,20,1386
sin_metricas,1521,155
sin_readme,123,123


## 3. Cobertura por tarea

In [4]:
sg, st = ap.summary(res_g), ap.summary(res_t)
cob = sg[["fichas", "con_metricas", "cobertura"]].join(
    st[["con_metricas", "cobertura"]], lsuffix="_grasp", rsuffix="_todas")
cob["ganancia"] = cob["con_metricas_todas"] - cob["con_metricas_grasp"]
display(cob.head(15))
print("Fichas que solo recupera 'todas':", int(cob["ganancia"].sum()))

/tmp/ipykernel_11665/1657819953.py:2: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  cob = sg[["fichas", "con_metricas", "cobertura"]].join(


,fichas,con_metricas_grasp,cobertura_grasp,con_metricas_todas,cobertura_todas,ganancia
pipeline_tag,,,,,,
text-classification,580,11,0.019,510,0.879,499
image-classification,301,0,0.000,282,0.937,282
summarization,272,0,0.000,245,0.901,245
token-classification,141,0,0.000,134,0.950,134
translation,132,0,0.000,120,0.909,120
text-generation,79,3,0.038,7,0.089,4
tabular-classification,34,0,0.000,30,0.882,30
NaN,33,0,0.000,21,0.636,21
tabular-regression,32,0,0.000,31,0.969,31


Fichas que solo recupera 'todas': 1366


## 4. Resultado final

Elige `USAR` después de comparar la cobertura de las dos configuraciones. Si la diferencia es grande en las tareas de AutoTrain (clasificación, resumen, traducción), usa `todas`; el límite de K es una restricción de mantenimiento, no de calidad.

In [5]:
USAR = "todas"          # "todas" o "grasp"
res = res_t if USAR == "todas" else res_g

out = df.merge(res[["modelId", "n_metricas", "metrics_from_body", "reglas_usadas"]],
               on="modelId", how="left")
out["metrics_source"] = out["metrics_from_body"].map(
    lambda x: "readme_body" if isinstance(x, str) and x not in ("", "[]") else None)
out.to_csv("../data/processed/HFCO2_2026_con_metricas_body.csv", index=False)

con = out["metrics_source"].notna().sum()
print(f"configuración: {USAR} | modelos con métricas del cuerpo: {con} de {len(out)}")
print("modelos con métricas utilizables (model-index):", len(cards))

configuración: todas | modelos con métricas del cuerpo: 1386 de 2057
modelos con métricas utilizables (model-index): 390


## 5. Muestra para inspección

Revisa algunas a mano antes de usar el resultado.

In [6]:
import json
con = res[res["estado"] == "con_metricas"]
m = con.sample(min(12, len(con)), random_state=7)
for _, r in m.iterrows():
    print(f"\n{r.modelId}  [{r.pipeline_tag}]  reglas: {r.reglas_usadas}")
    for h in json.loads(r.metrics_from_body)[:6]:
        print(f"   {h['metric']:24s} {h['value']}")
    print(f"   https://huggingface.co/{r.modelId}")


bsenker/swords-attentive_t5_v1  [summarization]  reglas: inline:rouge
   rouge1                   62.14
   rouge2                   13.128
   rougel                   61.331
   rougelsum                60.728
   https://huggingface.co/bsenker/swords-attentive_t5_v1

systash/autotrain-fake_news_fine_tuned_v4-38998102353  [text-classification]  reglas: inline:clf_basic
   accuracy                 0.983
   precision                0.986
   recall                   0.979
   auc                      0.998
   f1                       0.982
   https://huggingface.co/systash/autotrain-fake_news_fine_tuned_v4-38998102353

Tritkoman/GermantoNorthFrisian  [translation]  reglas: inline:bleu
   sacrebleu                40.859
   https://huggingface.co/Tritkoman/GermantoNorthFrisian

jrosenzw/autotrain-diabetes-detection-2-74371139581  [tabular-classification]  reglas: inline:clf_basic
   accuracy                 0.753
   precision                0.667
   recall                   0.593
   auc      